In [4]:
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

import matplotlib.pyplot as plt 
from matplotlib.backends.backend_pdf import PdfPages 
%matplotlib inline 

### 1. import data

In [5]:
train = pd.read_csv('train_data.csv')
test = pd.read_csv('test_data.csv')
val = pd.read_csv('val_data.csv')
c1 = ['Gender', 'Married', 'Age', 'CreditScore', 'Dependents', 'NumBankAccts', 'HasCrCard', 'EmergingMarketFund', 'RealEstate',
      'PrivateEquity', 'GovtBonds', 'CorpBonds', 'ETF Tech', 'ETF Health', 'ETF Med', 'Debt', 'Net Assets', 'Mortgage', 
      'EstimatedSalary', 'Portfolio Return', 'Diversification', 'BusinessOwner', 'Revenue', 'LifeInsurance', 
      'NumTransactions', 'DaysSinceLastTransaction', 'ForeignAssets', 'NumProducts']
c2 = [0, 0, -1, -1, -1, -1, -1, -1, -1,
      -1, -1, -1, -1, -1, -1, 1, -1, -1,
      0, -1, -1, -1, -1, -1, 
      -1, 1, -1, -1]

### 2. Random Forest

In [6]:
train_rf = train.copy()
train_rf['Gender'] = np.where(train_rf['Gender']=='Male',1,np.where(train_rf['Gender']=='Female',0,np.nan))
test_rf = test.copy()
test_rf['Gender'] = np.where(test_rf['Gender']=='Male',1,np.where(test_rf['Gender']=='Female',0,np.nan))

for i in train_rf.columns[1:-1]:
    j = np.round(np.mean(train_rf[i]),0)
    train_rf[i] = train_rf[i].fillna(j)
    test_rf[i] = test_rf[i].fillna(j)

ne1 = []
md1 = []
ml1 = []
mf1 = []
ms1 = []
ar1 = []
ar2 = []
for ne2 in tqdm([10,20,50], colour='white'):
    for md2 in [2,5,10]:
        for ml2 in [100,200,500]:
            for mf2 in [0.1,0.2,0.5]:
                for ms2 in [0.2,0.5,1.0]:
                    
                    ne1.append(ne2)
                    md1.append(md2)
                    ml1.append(ml2)
                    mf1.append(mf2)
                    ms1.append(ms2)
                    
                    x = train_rf.drop(['CustomerID', 'Churn'], axis=1)
                    y = train_rf[['Churn']]
                    model = RandomForestClassifier(criterion='gini', n_estimators=ne2, max_depth=md2, min_samples_leaf=ml2, 
                                                   max_features=mf2, max_samples=ms2, random_state=42, monotonic_cst=c2)
                    model.fit(x, y)
                    p = model.predict_proba(x)[:, 1]
                    ar1.append(roc_auc_score(y, p))
                    
                    x = test_rf.drop(['CustomerID', 'Churn'], axis=1)
                    y = test_rf[['Churn']]
                    p = model.predict_proba(x)[:, 1]
                    ar2.append(roc_auc_score(y, p))
        
gb = pd.DataFrame({'n_estimators':ne1, 'max_depth':md1, 'min_samples_leaf':ml1, 'max_features':mf1, 
                   'max_samples':ms1, 'train_auroc':ar1, 'test_auroc':ar2})
gb = gb.sort_values('test_auroc', ascending=False).head(15)
gb

100%|██████████| 3/3 [00:32<00:00, 10.71s/it]


,n_estimators,max_depth,min_samples_leaf,max_features,max_samples,train_auroc,test_auroc
91,20,2,200,0.1,0.5,0.526575,0.512574
118,20,5,200,0.1,0.5,0.531429,0.509546
145,20,10,200,0.1,0.5,0.531429,0.509546
82,20,2,100,0.1,0.5,0.528793,0.508657
163,50,2,100,0.1,0.5,0.536952,0.506124
172,50,2,200,0.1,0.5,0.531763,0.505491
226,50,10,200,0.1,0.5,0.536212,0.504857
199,50,5,200,0.1,0.5,0.536212,0.504857
90,20,2,200,0.1,0.2,0.516215,0.503587
120,20,5,200,0.2,0.2,0.521989,0.502695


In [7]:
train_rf = train.copy()
train_rf['Gender'] = np.where(train_rf['Gender']=='Male',1,np.where(train_rf['Gender']=='Female',0,np.nan))
test_rf = test.copy()
test_rf['Gender'] = np.where(test_rf['Gender']=='Male',1,np.where(test_rf['Gender']=='Female',0,np.nan))
val_rf = val.copy()
val_rf['Gender'] = np.where(val_rf['Gender']=='Male',1,np.where(val_rf['Gender']=='Female',0,np.nan))

for i in train_rf.columns[1:-1]:
    j = np.round(np.mean(train_rf[i]),0)
    train_rf[i] = train_rf[i].fillna(j)
    test_rf[i] = test_rf[i].fillna(j)
    val_rf[i] = val_rf[i].fillna(j)

x = train_rf.drop(['CustomerID', 'Churn'], axis=1)
y = train_rf[['Churn']]
model = RandomForestClassifier(criterion='gini', n_estimators=20, max_depth=2, min_samples_leaf=200, 
                               max_features=0.1, max_samples=0.5, random_state=42, monotonic_cst=c2)
model.fit(x, y)
p = model.predict_proba(x)[:, 1]
print('train shape',train_rf.shape,'\t\t train avg chrun rate',np.round(np.mean(train_rf['Churn'])*100,1),'\t\t train auroc',np.round(roc_auc_score(y, p),3)) 
        
x = test_rf.drop(['CustomerID', 'Churn'], axis=1)
y = test_rf[['Churn']]
p = model.predict_proba(x)[:, 1]
print('test shape',test_rf.shape,'\t\t test avg chrun rate',np.round(np.mean(test_rf['Churn'])*100,1),'\t\t test auroc',np.round(roc_auc_score(y, p),3)) 

x = val_rf.drop(['CustomerID', 'Churn'], axis=1)
y = val_rf[['Churn']]
p = model.predict_proba(x)[:, 1]
print('val shape',val_rf.shape,'\t\t val avg chrun rate',np.round(np.mean(val_rf['Churn'])*100,1),'\t\t val auroc',np.round(roc_auc_score(y, p),3)) 

train shape (5120, 30) 		 train avg chrun rate 50.2 		 train auroc 0.527
test shape (2462, 30) 		 test avg chrun rate 49.4 		 test auroc 0.513
val shape (2418, 30) 		 val avg chrun rate 49.0 		 val auroc 0.513


### 3. Light GBM

In [11]:
train_lgb = train.copy()
train_lgb['Gender'] = train_lgb['Gender'].astype('category')
test_lgb = test.copy()
test_lgb['Gender'] = test_lgb['Gender'].astype('category')

ne1, md1, ml1, mf1, ms1, ar1, ar2 = [], [], [], [], [], [], []

for ne2 in tqdm([2,5,10,20], colour='white'):
    for md2 in [2,5,10,20]:
        for ml2 in [100,200,500,1000]:
            for mf2 in [0.1,0.2,0.5,1.0]:      
                for ms2 in [0.1,0.2,0.5,1.0]:  
                    if mf2 == 1.0 and ms2 == 1.0: continue
                    ne1.append(ne2); md1.append(md2); ml1.append(ml2); 
                    mf1.append(mf2); ms1.append(ms2);

                    x = train_lgb.drop(['CustomerID', 'Churn'], axis=1)
                    y = train_lgb[['Churn']]
                    train_set = lgb.Dataset(x, label=y, categorical_feature=['Gender'], free_raw_data=False)

                    params = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'rf', 'max_depth': md2,
                                'min_data_in_leaf': ml2, 'feature_fraction': mf2, 'bagging_fraction': ms2,
                                'bagging_freq': 1, 'monotone_constraints': c2, 'monotone_constraints_method': 'advanced', 
                                'verbosity': -1, 'seed': 42}
                    model = lgb.train(params, train_set, num_boost_round=ne2) 
                    p = model.predict(x)
                    ar1.append(roc_auc_score(y, p))

                    x = test_lgb.drop(['CustomerID', 'Churn'], axis=1)
                    y = test_lgb[['Churn']]
                    p = model.predict(x)
                    ar2.append(roc_auc_score(y, p))

gb = pd.DataFrame({'n_estimators': ne1, 'max_depth': md1, 'min_data_in_leaf': ml1, 'feature_fraction': mf1, 
                   'bagging_fraction': ms1, 'train_auroc': ar1, 'test_auroc': ar2})
gb = gb.sort_values('test_auroc', ascending=False).head(15)
gb

100%|██████████| 4/4 [00:39<00:00,  9.88s/it]


,n_estimators,max_depth,min_data_in_leaf,feature_fraction,bagging_fraction,train_auroc,test_auroc
639,10,10,500,0.5,0.2,0.498406,0.515969
699,10,20,500,0.5,0.2,0.498406,0.515969
579,10,5,500,0.5,0.2,0.498406,0.515969
519,10,2,500,0.5,0.2,0.498406,0.515969
205,2,20,200,0.5,0.5,0.540936,0.514920
85,2,5,200,0.5,0.5,0.540936,0.514920
145,2,10,200,0.5,0.5,0.540936,0.514920
399,5,10,500,0.5,0.2,0.496538,0.514340
459,5,20,500,0.5,0.2,0.496538,0.514340
279,5,2,500,0.5,0.2,0.496538,0.514340


In [12]:
train_lgb = train.copy()
train_lgb['Gender'] = train_lgb['Gender'].astype('category')
test_lgb = test.copy()
test_lgb['Gender'] = test_lgb['Gender'].astype('category')
val_lgb = val.copy()
val_lgb['Gender'] = val_lgb['Gender'].astype('category')

x = train_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = train_lgb[['Churn']]
params = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'rf', 'max_depth': 10,
            'min_data_in_leaf': 500, 'feature_fraction': 0.5, 'bagging_fraction': 0.2,
            'bagging_freq': 1, 'monotone_constraints': c2, 'monotone_constraints_method': 'advanced', 
            'verbosity': -1, 'seed': 42}
model = lgb.train(params, train_set, num_boost_round=10) 
p = model.predict(x)
print('train shape',train_lgb.shape,'\t\t train avg chrun rate',np.round(np.mean(train_lgb['Churn'])*100,1),'\t\t train auroc',np.round(roc_auc_score(y, p),3)) 
        
x = test_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = test_lgb[['Churn']]
p = model.predict(x)
print('test shape',test_lgb.shape,'\t\t test avg chrun rate',np.round(np.mean(test_lgb['Churn'])*100,1),'\t\t test auroc',np.round(roc_auc_score(y, p),3)) 

x = val_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = val_lgb[['Churn']]
p = model.predict(x)
print('val shape',val_lgb.shape,'\t\t val avg chrun rate',np.round(np.mean(val_lgb['Churn'])*100,1),'\t\t val auroc',np.round(roc_auc_score(y, p),3)) 

train shape (5120, 30) 		 train avg chrun rate 50.2 		 train auroc 0.498
test shape (2462, 30) 		 test avg chrun rate 49.4 		 test auroc 0.516
val shape (2418, 30) 		 val avg chrun rate 49.0 		 val auroc 0.494
